Задание 1. Знакомство с данными

In [1]:
import sqlite3
import pandas as pd

# Подключение к базе
conn = sqlite3.connect("w2_database.sqlite")

# Загрузка всех таблиц
exchange   = pd.read_sql("SELECT * FROM exchange",   conn)
market     = pd.read_sql("SELECT * FROM market",     conn)
instrument = pd.read_sql("SELECT * FROM instrument", conn)
currency   = pd.read_sql("SELECT * FROM currency",   conn)

conn.close()

# Структура каждой таблицы
for name, df in [("exchange", exchange), ("market", market),
                 ("instrument", instrument), ("currency", currency)]:
    print(f"=== {name} ({len(df)} строк) ===")
    df.info()
    print()

=== exchange (290 строк) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          290 non-null    int64  
 1   name        290 non-null    object 
 2   cmc_id      150 non-null    float64
 3   foundation  290 non-null    object 
 4   country     290 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 11.5+ KB

=== market (16916 строк) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16916 entries, 0 to 16915
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             16916 non-null  int64 
 1   symbol         16916 non-null  object
 2   instrument_id  16916 non-null  int64 
 3   exchange_id    16916 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 528.8+ KB

=== instrument (7814 строк) ===
<class 'pandas.core.frame.DataFrame'

Задание 2. Типы валют

In [3]:
# Приводим is_fiat к булевому типу
currency["is_fiat"] = currency["is_fiat"].astype(bool)

# Приводим is_stable (NaN считаем False)
currency["is_stable"] = currency["is_stable"].map({"TRUE": True, "FALSE": False}).fillna(False)

# Фиатные валюты
fiat = currency[currency["is_fiat"]]
print(f"Фиатные ({len(fiat)}):")
print(fiat)

# Стейблкоины
stable = currency[currency["is_stable"]]
print(f"\nСтейблкоины ({len(stable)}):")
print(stable)

# Пересечение
overlap = currency[currency["is_fiat"] & currency["is_stable"]]
print(f"\nПересечение ({len(overlap)}):")
print(overlap if len(overlap) > 0 else "Пересечений нет")

Фиатные (146):
           id                    name symbol  cmc_id  is_fiat  is_stable
1523     2649            Turkish Lira    TRY   99999     True      False
2542     4857          Russian Rouble    RUB   99999     True      False
2575     5004                    Euro    EUR   99999     True      False
2576     5015                    Rand    ZAR   99999     True      False
2577     5019  British Pound Sterling    GBP   99999     True      False
...       ...                     ...    ...     ...      ...        ...
4881  1320770                     Sol    PEN   99999     True      False
4882  1320771                 Bolivar    VES   99999     True      False
4883  1320772                    Tala    WST   99999     True      False
4884  1320773  Solomon Islands Dollar    SBD   99999     True      False
4885  1320774                 Paʻanga    TOP   99999     True      False

[146 rows x 6 columns]

Стейблкоины (0):
Empty DataFrame
Columns: [id, name, symbol, cmc_id, is_fiat, is_sta

/tmp/ipykernel_5764/960968809.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  currency["is_stable"] = currency["is_stable"].map({"TRUE": True, "FALSE": False}).fillna(False)


Задание 3. Ошибки в символах инструментов

In [4]:
# символы base и quote валют
instrument_check = instrument.merge(
    currency[["id", "symbol"]].rename(columns={"id": "base_currency_id", "symbol": "base_symbol"}),
    on="base_currency_id"
).merge(
    currency[["id", "symbol"]].rename(columns={"id": "quote_currency_id", "symbol": "quote_symbol"}),
    on="quote_currency_id"
)

# ожидаемый символ = конкатенация base + quote
instrument_check["expected_symbol"] = instrument_check["base_symbol"] + instrument_check["quote_symbol"]

# инструменты с расхождением
errors = instrument_check[instrument_check["symbol"] != instrument_check["expected_symbol"]]
print(f"Ошибок найдено: {len(errors)}")
print(errors[["id", "symbol", "expected_symbol"]].head(10))

# замена symbol на ожидаемый
instrument.loc[instrument["id"].isin(errors["id"]), "symbol"] = (
    errors.set_index("id")["expected_symbol"]
)

print("\nПосле исправления ошибок:",
      (instrument.merge(instrument_check[["id", "expected_symbol"]], on="id")
       .pipe(lambda df: (df["symbol"] != df["expected_symbol"]).sum())), "ошибок")

Ошибок найдено: 118
        id       symbol expected_symbol
55     159      KNCUSDT        KNCLUSDT
151    456       KNCETH         KNCLETH
167    497   TONCOINBTC          TONBTC
180    533       KNCBTC         KNCLBTC
389   1150  TONCOINUSDT         TONUSDT
515   1989       PAXBTC         USDPBTC
516   1991      PAXUSDT        USDPUSDT
866   3224       KNCUSD         KNCLUSD
883   3256       PAXUSD         USDPUSD
1011  3491       KNCEUR         KNCLEUR

После исправления ошибок: 118 ошибок


Задание 4. Мастер-таблица

In [5]:
database = (
    market
    # Биржа
    .merge(exchange[["id", "name", "country"]].rename(columns={
        "id": "exchange_id", "name": "exchange_name"
    }), on="exchange_id")
    # Инструмент
    .merge(instrument[["id", "base_currency_id", "quote_currency_id"]].rename(columns={
        "id": "instrument_id"
    }), on="instrument_id")
    # Base-валюта
    .merge(currency[["id", "symbol"]].rename(columns={
        "id": "base_currency_id", "symbol": "base_symbol"
    }), on="base_currency_id")
    # Quote-валюта
    .merge(currency[["id", "symbol", "is_fiat", "is_stable"]].rename(columns={
        "id": "quote_currency_id", "symbol": "quote_symbol"
    }), on="quote_currency_id")
)

print(f"Строк в мастер-таблице: {len(database)}")
print(database[["symbol", "exchange_name", "country", "base_symbol", "quote_symbol", "is_fiat", "is_stable"]].head())

Строк в мастер-таблице: 16916
    symbol exchange_name country base_symbol quote_symbol  is_fiat  is_stable
0   ETHBTC       Binance   Malta         ETH          BTC    False      False
1   LTCBTC       Binance   Malta         LTC          BTC    False      False
2   BNBBTC       Binance   Malta         BNB          BTC    False      False
3   NEOBTC       Binance   Malta         NEO          BTC    False      False
4  QTUMETH       Binance   Malta        QTUM          ETH    False      False


Задание 5. Неактивные инструменты и валюты

In [6]:
# Инструменты, которые не связаны ни с одним маркетом
inactive_instruments = instrument[~instrument["id"].isin(database["instrument_id"])]
print(f"Инструменты без маркетов: {len(inactive_instruments)}")
print(inactive_instruments)

# Валюты, которых нет ни как base, ни как quote в мастер-таблице
active_currency_ids = set(database["base_currency_id"]) | set(database["quote_currency_id"])

inactive_currencies = currency[~currency["id"].isin(active_currency_ids)]
print(f"\nВалюты без маркетов: {len(inactive_currencies)}")
print(inactive_currencies)

Инструменты без маркетов: 3
         id    symbol  base_currency_id  quote_currency_id
5367  46752   BTCUSDQ                 1               4133
6680  49955  XAUTUSDQ              5109            1545183
6911  50418   BTCUSDQ                 1            1545183

Валюты без маркетов: 2298
           id                    name symbol  cmc_id  is_fiat  is_stable
25         26                   Bytom    BTM    1866    False      False
27         29                Populous    PPT    1789    False      False
28         30                Bytecoin    BCN     372    False      False
34         36         Bitcoin Private   BTCP    2575    False      False
35         37                DigixDAO    DGD    1229    False      False
...       ...                     ...    ...     ...      ...        ...
4881  1320770                     Sol    PEN   99999     True      False
4882  1320771                 Bolivar    VES   99999     True      False
4883  1320772                    Tala    WST   99999

Задание 6. Анализ quote-валют

In [7]:
# а) Классификация валют по роли
base_ids  = set(database["base_currency_id"])
quote_ids = set(database["quote_currency_id"])

def classify_role(cid):
    in_base  = cid in base_ids
    in_quote = cid in quote_ids
    if in_base and in_quote:
        return "both"
    elif in_base:
        return "only_base"
    else:
        return "only_quote"

currency["role"] = currency["id"].apply(classify_role)
print("Валют по категориям:")
print(currency["role"].value_counts())

# б) Сабсет валют, которые встречаются как quote
quote_currencies = currency[currency["id"].isin(quote_ids)].copy()

# в) Добавляем тип quote-валюты
import numpy as np

quote_currencies["type"] = np.select(
    [quote_currencies["is_fiat"], quote_currencies["is_stable"]],
    ["fiat_quote", "stable_quote"],
    default="crypto_quote"
)
print("\nQuote-валюты по типу:")
print(quote_currencies["type"].value_counts())

# г) Статистика по каждой quote-валюте
quote_stats = (
    database
    .groupby("quote_symbol")
    .agg(
        markets      =("symbol", "count"),
        exchanges    =("exchange_id", "nunique"),
        base_currencies=("base_currency_id", "nunique")
    )
    .sort_values("markets", ascending=False)
    .reset_index()
)
print("\nСтатистика quote-валют:")
print(quote_stats)

# д) Топ-5 стейблкоинов по числу уникальных бирж
print("\nТоп-5 стейблкоин quote-валют по числу бирж:")
print(
    database[database["is_stable"]]
    .groupby("quote_symbol")
    .agg(exchanges=("exchange_id", "nunique"))
    .sort_values("exchanges", ascending=False)
    .head(5)
    .reset_index()
)

Валют по категориям:
role
only_base     4336
only_quote    2325
both            35
Name: count, dtype: int64

Quote-валюты по типу:
type
crypto_quote    38
fiat_quote      24
Name: count, dtype: int64

Статистика quote-валют:
   quote_symbol  markets  exchanges  base_currencies
0          USDT    11036         17             4243
1           USD     1761          7              884
2          USDC      906         14              397
3           EUR      789          9              556
4           KRW      649          2              433
..          ...      ...        ...              ...
57          NGN        1          1                1
58         USDG        1          1                1
59          UAH        1          1                1
60         XAUT        1          1                1
61          XRP        1          1                1

[62 rows x 4 columns]

Топ-5 стейблкоин quote-валют по числу бирж:
Empty DataFrame
Columns: [quote_symbol, exchanges]
Index: []


Задание 7. Профиль каждой биржи

In [8]:
exchange_profile = (
    database
    .groupby("exchange_name")
    .agg(
        total_markets      =("symbol", "count"),
        unique_bases       =("base_symbol", "nunique"),
        unique_quotes      =("quote_symbol", "nunique"),
        fiat_quote_count   =("is_fiat", "sum"),
        stable_quote_count =("is_stable", "sum"),
        fiat_quote_list    =("quote_symbol", lambda x: ", ".join(
            sorted(database.loc[x.index[database.loc[x.index, "is_fiat"]], "quote_symbol"].unique())
        ))
    )
    .reset_index()
    .sort_values("total_markets", ascending=False)
)

print(exchange_profile)

   exchange_name  total_markets  unique_bases  unique_quotes  \
12          MEXC           2431          2035              9   
6        Gate.io           2083          1900              6   
0        Binance           1711           501             37   
16        XT.com           1357          1248             10   
1        BitMart           1349          1289              8   
8         Kraken           1252           573             19   
9         Kucoin           1129           930             12   
11         Lbank            984           955              7   
13           OKX            740           294             11   
10       Latoken            725           707              7   
5     Crypto.com            525           366              8   
4       Coinbase            493           375              7   
14      Poloniex            485           445              7   
17       bithumb            427           423              2   
15         Upbit            395         

Задание 8. Рейтинг важности бирж

In [10]:
# Уникальные пары биржа-base
db_unique = database[["exchange_name", "base_symbol"]].drop_duplicates()

# Для каждой base-валюты считаем кол-во бирж
db_unique["exchange_count"] = (
    db_unique.groupby("base_symbol")["exchange_name"]
    .transform("nunique")
)

# Вес по числу бирж согласно формуле
db_unique["weight"] = np.select(
    [
        db_unique["exchange_count"] == 1,
        db_unique["exchange_count"] == 2,
        db_unique["exchange_count"].between(3, 5),
        db_unique["exchange_count"] >= 6,
    ],
    [1.0, 0.5, 0.2, 0.05],
    default=0.0
)

exchange_score = (
    db_unique
    .groupby("exchange_name")["weight"]
    .sum()
    .reset_index()
    .rename(columns={"weight": "score"})
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print(exchange_score)

   exchange_name   score
0           MEXC  710.15
1        Gate.io  663.35
2        BitMart  450.05
3        Latoken  341.40
4          Lbank  336.75
5         XT.com  323.70
6       Poloniex  149.30
7         Kucoin  118.70
8         Kraken   89.55
9   Huobi Global   86.15
10       Binance   72.85
11    Crypto.com   50.55
12       bithumb   42.70
13      Coinbase   32.15
14         Upbit   20.85
15      Bitfinex   19.65
16           OKX   19.55
17      Bitstamp   11.65
